**Step 1: Import libraries and upload the CSV file**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

df = pd.read_csv("heart_attack_dataset.csv")
print(df.head())
# Keep an untouched copy for before-imputation analysis
df_raw = df.copy()

os.makedirs("plots", exist_ok=True)

print("Dataset shape:", df.shape)

**Task 2 — Null Value Analysis and Median Imputation**

In [ ]:
# Keep a copy before imputation
df_before_imputation = df.copy()

# Missing-value count and percentage
missing_count = df.isnull().sum()

missing_percentage = (
    df.isnull().sum() / df.shape[0]
) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

print("Missing-value summary:")
display(missing_summary)

# Columns with more than 20% missing values
columns_above_20 = missing_summary[
    missing_summary["Missing Percentage"] > 20
]

print("Columns exceeding 20% missing values:")

if columns_above_20.empty:
    print("No columns exceed 20% missing values.")
else:
    display(columns_above_20)

# Fill numeric columns below or equal to 20% missing values
numeric_columns = df.select_dtypes(
    include=np.number
).columns

for col in numeric_columns:

    missing_pct = (
        df[col].isnull().sum()
        / df.shape[0]
    ) * 100

    if 0 < missing_pct <= 20:

        print(
            col,
            "Mean:",
            round(df[col].mean(), 2),
            "Median:",
            round(df[col].median(), 2)
        )

        df[col] = df[col].fillna(
            df[col].median()
        )

print("\nMissing values after numeric imputation:")
display(df.isnull().sum())

**Task 3 — Duplicate Detection and Removal**

In [ ]:
# Missing percentages before duplicate removal
null_percentage_before_duplicates = (
    df.isnull().sum() / df.shape[0]
) * 100

# Count duplicate rows
duplicate_count = df.duplicated().sum()

print("Duplicate rows found:", duplicate_count)

rows_before = df.shape[0]

# Remove duplicate rows
df = df.drop_duplicates().copy()

rows_after = df.shape[0]
rows_removed = rows_before - rows_after

print("Rows before duplicate removal:", rows_before)
print("Rows after duplicate removal:", rows_after)
print("Rows removed:", rows_removed)

null_percentage_after_duplicates = (
    df.isnull().sum() / df.shape[0]
) * 100

duplicate_null_comparison = pd.DataFrame({
    "Before Duplicate Removal (%)":
        null_percentage_before_duplicates,
    "After Duplicate Removal (%)":
        null_percentage_after_duplicates
})

duplicate_null_comparison["Change (%)"] = (
    duplicate_null_comparison[
        "After Duplicate Removal (%)"
    ]
    -
    duplicate_null_comparison[
        "Before Duplicate Removal (%)"
    ]
)

display(duplicate_null_comparison)
changed_columns = duplicate_null_comparison[
    duplicate_null_comparison["Change (%)"].abs() > 0.000001
]

if changed_columns.empty:
    print(
        "Duplicate removal did not change any "
        "column's null percentage."
    )
else:
    print("Columns with changed null percentages:")
    display(changed_columns)

**Task 4 — Data Type Correction**

In [ ]:
# Memory usage before conversion
memory_before = df.memory_usage(deep=True).sum()

print("Memory usage before conversion:")
print(memory_before, "bytes")
# Convert identifier to string
df["patient_id"] = df["patient_id"].astype("string")

# Repetitive categorical columns
categorical_columns = [
    "gender",
    "chest_pain_type",
    "resting_ecg",
    "st_slope",
    "thalassemia",
    "smoking_status",
    "alcohol_consumption",
    "physical_activity"
]

for col in categorical_columns:
    if col in df.columns:
        df[col] = df[col].astype("category")
        memory_after = df.memory_usage(deep=True).sum()

memory_saved = memory_before - memory_after

memory_reduction_percentage = (
    memory_saved / memory_before
) * 100

print("Memory usage after conversion:")
print(memory_after, "bytes")

print("Memory saved:")
print(memory_saved, "bytes")

print(
    "Memory reduction percentage:",
    round(memory_reduction_percentage, 2),
    "%"
)

print("\nUpdated data types:")
print(df.dtypes)

**Task 5 — Descriptive Statistics and Skewness**

In [ ]:
# Select numeric columns
numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numeric columns:")
print(numeric_columns)
print("Descriptive statistics:")
display(df[numeric_columns].describe().T)
skewness = df[numeric_columns].skew()

skewness_table = pd.DataFrame({
    "Skewness": skewness,
    "Absolute Skewness": skewness.abs()
}).sort_values(
    "Absolute Skewness",
    ascending=False
)

print("Skewness table:")
display(skewness_table)
most_skewed_column = skewness_table.index[0]

most_skewed_value = skewness_table.loc[
    most_skewed_column,
    "Skewness"
]

print("Most skewed numeric column:")
print(most_skewed_column)

print("Skewness value:")
print(round(most_skewed_value, 4))

if most_skewed_value > 0:
    print("The distribution is positively skewed.")
elif most_skewed_value < 0:
    print("The distribution is negatively skewed.")
else:
    print("The distribution is approximately symmetric.")

**Task 6 — Imputation Strategy Comparison**

In [ ]:
top_two_skewed_columns = (
    skewness.abs()
    .sort_values(ascending=False)
    .head(2)
    .index
    .tolist()
)

print("Two columns with highest absolute skewness:")
print(top_two_skewed_columns)
imputation_comparison = []

for col in top_two_skewed_columns:

    imputation_comparison.append({
        "Column": col,
        "Skewness": skewness[col],
        "Mean Before Imputation": df[col].mean(),
        "Median Before Imputation": df[col].median(),
        "Missing Values Before Imputation":
            df[col].isnull().sum()
    })

imputation_comparison_df = pd.DataFrame(
    imputation_comparison
)

print("Mean and median before imputation:")
display(imputation_comparison_df)
numeric_imputation_details = []

for col in df.select_dtypes(include=np.number).columns:

    missing_count = df[col].isnull().sum()

    missing_percentage = (
        missing_count / df.shape[0]
    ) * 100

    if 0 < missing_percentage <= 20:

        median_value = df[col].median()

        df[col] = df[col].fillna(
            median_value
        )

        numeric_imputation_details.append({
            "Column": col,
            "Missing Percentage":
                missing_percentage,
            "Median Used": median_value,
            "Values Filled": missing_count
        })

print("Numeric median-imputation details:")
display(pd.DataFrame(numeric_imputation_details))
print(
    "Remaining missing values in the two "
    "most skewed columns:"
)

display(
    df[top_two_skewed_columns]
    .isnull()
    .sum()
    .to_frame("Remaining Null Values")
)

**Task 7 — IQR Outlier Detection**

In [ ]:
iqr_columns = [
    "cholesterol",
    "resting_bp"
]

iqr_results = []

for col in iqr_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_mask = (
        (df[col] < lower_bound)
        |
        (df[col] > upper_bound)
    )

    outlier_count = outlier_mask.sum()

    iqr_results.append({
        "Column": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outlier_count
    })

iqr_results_df = pd.DataFrame(
    iqr_results
)

print("IQR outlier analysis:")
display(iqr_results_df)

**Task 8 — Visualizations**

**Line Plot**

In [ ]:

plt.figure(figsize=(12, 6))

plt.plot(
    df.index,
    df["max_heart_rate"]
)

plt.title(
    "Maximum Heart Rate Across Patient Records"
)

plt.xlabel("Row Index")
plt.ylabel("Maximum Heart Rate")

plt.tight_layout()

plt.savefig(
    "plots/line_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()



Bar Chart **bold text**

In [ ]:
bar_data = (
    df.groupby(
        "chest_pain_type",
        observed=False
    )["heart_attack_risk"]
    .mean()
    .sort_values(ascending=False)
)

display(
    bar_data.to_frame(
        "Mean Heart Attack Risk"
    )
)
plt.figure(figsize=(10, 6))

plt.bar(
    bar_data.index.astype(str),
    bar_data.values
)

plt.title(
    "Mean Heart Attack Risk by Chest Pain Type"
)

plt.xlabel("Chest Pain Type")
plt.ylabel("Mean Heart Attack Risk")

plt.xticks(
    rotation=30,
    ha="right"
)

plt.tight_layout()

plt.savefig(
    "plots/bar_chart.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

**Histogram**

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x=most_skewed_column,
    bins=20,
    kde=True
)

plt.title(
    f"Distribution of "
    f"{most_skewed_column.replace('_', ' ').title()}"
)

plt.xlabel(
    most_skewed_column
    .replace("_", " ")
    .title()
)

plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    "plots/histogram.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

**Scatter Plot**

In [ ]:
scatter_correlation = df[
    ["age", "max_heart_rate"]
].corr().iloc[0, 1]

print(
    "Correlation between age and maximum heart rate:",
    round(scatter_correlation, 4)
)
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="age",
    y="max_heart_rate",
    alpha=0.6
)

plt.title(
    "Age vs Maximum Heart Rate"
)

plt.xlabel("Age")
plt.ylabel("Maximum Heart Rate")

plt.tight_layout()

plt.savefig(
    "plots/scatter_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
absolute_correlation = abs(
    scatter_correlation
)

if absolute_correlation < 0.20:
    scatter_strength = "very weak"
elif absolute_correlation < 0.40:
    scatter_strength = "weak"
elif absolute_correlation < 0.60:
    scatter_strength = "moderate"
elif absolute_correlation < 0.80:
    scatter_strength = "strong"
else:
    scatter_strength = "very strong"

if scatter_correlation > 0:
    scatter_direction = "positive"
elif scatter_correlation < 0:
    scatter_direction = "negative"
else:
    scatter_direction = "no"

print(
    "Relationship:",
    scatter_strength,
    scatter_direction
)

**Box Plot**

In [ ]:
plt.figure(figsize=(9, 6))

sns.boxplot(
    data=df,
    x="heart_attack_risk",
    y="max_heart_rate"
)

plt.title(
    "Maximum Heart Rate by Heart Attack Risk"
)

plt.xlabel("Heart Attack Risk")
plt.ylabel("Maximum Heart Rate")

plt.tight_layout()

plt.savefig(
    "plots/box_plot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
box_plot_summary = (
    df.groupby(
        "heart_attack_risk"
    )["max_heart_rate"]
    .agg([
        "median",
        "mean",
        "std",
        "count"
    ])
)

display(box_plot_summary)

**Task 9 — Pearson Correlation and Heat Map**

In [ ]:
numeric_df = df.select_dtypes(
    include=np.number
).copy()
pearson_matrix = numeric_df.corr(
    method="pearson"
)

print("Pearson correlation matrix:")
display(pearson_matrix)
plt.figure(figsize=(16, 12))

sns.heatmap(
    pearson_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title(
    "Pearson Correlation Heat Map"
)

plt.tight_layout()

plt.savefig(
    "plots/correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
absolute_pearson = pearson_matrix.abs().copy()

# Remove diagonal values
np.fill_diagonal(
    absolute_pearson.values,
    np.nan
)

# Keep upper triangle only
upper_triangle = absolute_pearson.where(
    np.triu(
        np.ones(
            absolute_pearson.shape
        ),
        k=1
    ).astype(bool)
)

correlation_pairs = (
    upper_triangle
    .stack()
    .sort_values(ascending=False)
)

highest_pair = correlation_pairs.index[0]

highest_column_1 = highest_pair[0]
highest_column_2 = highest_pair[1]

highest_correlation_value = pearson_matrix.loc[
    highest_column_1,
    highest_column_2
]

print(
    "Highest absolute correlation pair:",
    highest_column_1,
    "and",
    highest_column_2
)

print(
    "Correlation value:",
    round(highest_correlation_value, 4)
)

**Task 10 — Spearman Rank Correlation**

In [ ]:
spearman_matrix = numeric_df.corr(
    method="spearman"
)

print("Spearman correlation matrix:")
display(spearman_matrix)
print("Pearson correlation matrix:")
display(pearson_matrix)
difference_matrix = (
    spearman_matrix - pearson_matrix
).abs()

print(
    "Absolute difference matrix: "
    "|Spearman - Pearson|"
)

display(difference_matrix)
correlation_difference_rows = []

columns = difference_matrix.columns.tolist()

for i in range(len(columns)):

    for j in range(i + 1, len(columns)):

        col1 = columns[i]
        col2 = columns[j]

        pearson_value = pearson_matrix.loc[
            col1,
            col2
        ]

        spearman_value = spearman_matrix.loc[
            col1,
            col2
        ]

        absolute_difference = abs(
            spearman_value - pearson_value
        )

        if abs(spearman_value) > abs(pearson_value):

            relationship_type = (
                "Monotonic but potentially non-linear"
            )

            preferred_measure = "Spearman"

        else:

            relationship_type = (
                "Approximately linear or "
                "Pearson is at least as strong"
            )

            preferred_measure = "Pearson"

        correlation_difference_rows.append({
            "Column 1": col1,
            "Column 2": col2,
            "Pearson": pearson_value,
            "Spearman": spearman_value,
            "Absolute Difference":
                absolute_difference,
            "Interpretation":
                relationship_type,
            "Preferred Measure":
                preferred_measure
        })
        correlation_difference_table = pd.DataFrame(
    correlation_difference_rows
).sort_values(
    "Absolute Difference",
    ascending=False
)

top_three_differences = (
    correlation_difference_table
    .head(3)
    .reset_index(drop=True)
)

print(
    "Three pairs with the largest "
    "|Spearman - Pearson| differences:"
)

display(top_three_differences)


**Task 11 — Grouped Aggregation**

In [ ]:
grouped_result = (
    df.groupby(
        "chest_pain_type",
        observed=False
    )["heart_attack_risk"]
    .agg([
        "mean",
        "std",
        "count"
    ])
)

print(
    "Heart attack risk grouped "
    "by chest pain type:"
)

display(grouped_result)
highest_mean_group = (
    grouped_result["mean"].idxmax()
)

highest_std_group = (
    grouped_result["std"].idxmax()
)

highest_mean = grouped_result["mean"].max()
lowest_mean = grouped_result["mean"].min()

if lowest_mean == 0:
    mean_ratio = np.inf
else:
    mean_ratio = highest_mean / lowest_mean

print(
    "Group with highest mean:",
    highest_mean_group
)

print(
    "Group with highest standard deviation:",
    highest_std_group
)

print(
    "Highest group mean:",
    round(highest_mean, 4)
)

print(
    "Lowest group mean:",
    round(lowest_mean, 4)
)

if np.isinf(mean_ratio):

    print(
        "Mean ratio is infinite because "
        "the lowest mean is zero."
    )

else:

    print(
        "Highest-to-lowest mean ratio:",
        round(mean_ratio, 4)
    )

**Task 12 — Fill Remaining Categorical Null Values**

In [ ]:
categorical_columns = df.select_dtypes(
    include=[
        "object",
        "string",
        "category"
    ]
).columns.tolist()

categorical_imputation_details = []

for col in categorical_columns:

    missing_count = df[col].isnull().sum()

    if missing_count > 0:

        mode_value = df[col].mode(
            dropna=True
        ).iloc[0]

        df[col] = df[col].fillna(
            mode_value
        )

        categorical_imputation_details.append({
            "Column": col,
            "Mode Used": str(mode_value),
            "Values Filled": missing_count
        })

print("Categorical imputation details:")
display(
    pd.DataFrame(
        categorical_imputation_details
    )
)
final_missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage":
        (
            df.isnull().sum()
            / df.shape[0]
        ) * 100
})

print("Final missing-value summary:")
display(final_missing_summary)

**Task 13 — Save the Cleaned Dataset**

In [ ]:
df.to_csv(
    "cleaned_data.csv",
    index=False
)

print(
    "cleaned_data.csv saved successfully."
)

print(
    "Final dataset shape:",
    df.shape
)
from google.colab import files

files.download(
    "cleaned_data.csv"
)